In [1]:
import torch

torch.__version__

'2.5.1+cu124'

In [2]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms

In [3]:
torch.manual_seed(4242)

In [4]:
train_loader = torch.utils.data.DataLoader(
    datasets.MNIST(
        # "../data/p1ch2/mnist",
        "./data/p1ch2/mnist",
        train=True,
        download=True,
        transform=transforms.Compose(
            [transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))]
        ),
    ),
    batch_size=64,
    shuffle=True,
)

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 9.91M/9.91M [00:00<00:00, 104MB/s]


Extracting ./data/p1ch2/mnist/MNIST/raw/train-images-idx3-ubyte.gz to ./data/p1ch2/mnist/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 28.9k/28.9k [00:00<00:00, 9.02MB/s]


Extracting ./data/p1ch2/mnist/MNIST/raw/train-labels-idx1-ubyte.gz to ./data/p1ch2/mnist/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 1.65M/1.65M [00:00<00:00, 102MB/s]


Extracting ./data/p1ch2/mnist/MNIST/raw/t10k-images-idx3-ubyte.gz to ./data/p1ch2/mnist/MNIST/raw

Failed to download (trying next):
HTTP Error 404: Not Found



100%|██████████| 4.54k/4.54k [00:00<00:00, 3.89MB/s]


Extracting ./data/p1ch2/mnist/MNIST/raw/t10k-labels-idx1-ubyte.gz to ./data/p1ch2/mnist/MNIST/raw



In [5]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.conv1 = nn.Conv2d(1, 10, kernel_size=5)
        self.conv2 = nn.Conv2d(10, 20, kernel_size=5)
        self.conv2_drop = nn.Dropout2d()
        self.fc1 = nn.Linear(320, 50)
        self.fc2 = nn.Linear(50, 10)

    def forward(self, x):
        x = F.relu(F.max_pool2d(self.conv1(x), 2))
        x = F.relu(F.max_pool2d(self.conv2_drop(self.conv2(x)), 2))
        x = x.view(-1, 320)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, training=self.training)
        x = self.fc2(x)
        return F.log_softmax(x, dim=1)

In [6]:
model = Net()

In [7]:
optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.5)

In [8]:
for epoch in range(10):
    for batch_idx, (data, target) in enumerate(train_loader):
        optimizer.zero_grad()
        output = model(data)
        loss = F.nll_loss(output, target)
        loss.backward()
        optimizer.step()
    print("Current loss", float(loss))

Current loss 0.4900275468826294
Current loss 0.12471085041761398
Current loss 0.429616779088974
Current loss 0.5233688950538635
Current loss 0.213942751288414
Current loss 0.2640897333621979
Current loss 0.2459457665681839
Current loss 0.24523641169071198
Current loss 0.031336452811956406
Current loss 0.06825054436922073


In [9]:
# torch.save(model.state_dict(), "../data/p1ch2/mnist/mnist.pth")
torch.save(model.state_dict(), "./data/p1ch2/mnist/mnist.pth")

In [11]:
pretrained_model = Net()
# pretrained_model.load_state_dict(torch.load("../data/p1ch2/mnist/mnist.pth"))
pretrained_model.load_state_dict(torch.load("./data/p1ch2/mnist/mnist.pth"))

<ipython-input-11-448887552d20>:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  pretrained_model.load_state_dict(torch.load("./data/p1ch2/mnist/mnist.pth"))


<All keys matched successfully>